# Large File Compression for GitHub

## Overview

This compression script automatically scans your entire codebase (all folders and subfolders) and compresses any file larger than 100 MB into a `.zip` file. The original file is then deleted, keeping your repository clean and GitHub-compatible.

## Why Is This Needed?

### GitHub's 100 MB File Limit

GitHub blocks individual files larger than **100 MB** by default. If you try to push a file exceeding this limit, GitHub will reject the push and your commit will fail.

In [1]:
# Cell 1: Import libraries
import os
import zipfile
from pathlib import Path

LOG_HEADER = "Original path -> Final path"

def ensure_compression_log(log_file="compression.txt"):
    """Create/update log file and ensure required header is first line."""
    log_path = Path(log_file)

    if not log_path.exists():
        log_path.write_text(LOG_HEADER + "\n", encoding="utf-8")
        return log_path

    lines = log_path.read_text(encoding="utf-8").splitlines()
    if not lines:
        lines = [LOG_HEADER]
    elif lines[0].strip() != LOG_HEADER:
        lines[0] = LOG_HEADER

    log_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return log_path

def append_compression_log(original_path, final_path, log_file="compression.txt"):
    """Append a single original->final mapping to the compression log."""
    log_path = ensure_compression_log(log_file)
    with log_path.open("a", encoding="utf-8") as f:
        f.write(f"{Path(original_path).resolve()} -> {Path(final_path).resolve()}\n")

def get_file_size_mb(file_path):
    """Get file size in MB"""
    return os.path.getsize(file_path) / (1024 * 1024)

def compress_file(file_path, delete_original=True):
    """Compress a single file to .zip and return output path on success."""
    try:
        file_path = Path(file_path)
        size_mb = get_file_size_mb(file_path)

        zip_path = file_path.with_name(f"{file_path.name}.zip")

        # Compress with max compression
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=9) as zipf:
            zipf.write(file_path, arcname=file_path.name)

        zip_size_mb = get_file_size_mb(zip_path)
        compression_ratio = ((size_mb - zip_size_mb) / size_mb) * 100 if size_mb > 0 else 0

        print(f"✓ {file_path.name}")
        print(f"  {size_mb:.2f}MB → {zip_size_mb:.2f}MB (-{compression_ratio:.1f}%)")

        if delete_original:
            file_path.unlink()
            print("  Original deleted")

        return zip_path
    except Exception as e:
        print(f"✗ Error: {file_path.name} - {e}")
        return None

def scan_and_compress(start_path=".", size_threshold_mb=100, delete_originals=True, dry_run=False):
    """
    Recursively scan directory and compress files larger than threshold
    """
    large_files = []
    total_size = 0

    print(f"🔍 Scanning {start_path} for files > {size_threshold_mb}MB...\n")

    # Recursively find all files
    for root, dirs, files in os.walk(start_path):
        # Skip common directories
        dirs[:] = [d for d in dirs if d not in {'.git', '__pycache__', '.venv', 'venv', 'node_modules', '.github'}]

        for file in files:
            file_path = Path(root) / file

            try:
                size_mb = get_file_size_mb(file_path)

                if size_mb > size_threshold_mb:
                    large_files.append((file_path, size_mb))
                    total_size += size_mb
                    print(f"  Found: {file_path} ({size_mb:.2f}MB)")

            except Exception as e:
                print(f"  Error reading {file_path}: {e}")

    if not large_files:
        print(f"✓ No files larger than {size_threshold_mb}MB found!")
        return

    print(f"\n{'='*60}")
    print("📊 Summary:")
    print(f"   Files found: {len(large_files)}")
    print(f"   Total size: {total_size:.2f}MB")
    print(f"{'='*60}\n")

    if dry_run:
        print("⚠️  DRY RUN MODE - Files will NOT be compressed\n")
        return

    # Ensure log file exists with correct header before appending mappings
    ensure_compression_log("compression.txt")

    # Compress each file
    print("🗜️  Compressing files...\n")
    compressed = 0
    failed = 0

    for file_path, size_mb in large_files:
        zip_path = compress_file(file_path, delete_original=delete_originals)
        if zip_path:
            compressed += 1
            append_compression_log(file_path, zip_path, log_file="compression.txt")
        else:
            failed += 1
        print()

    print(f"{'='*60}")
    print(f"✅ Complete: {compressed} compressed, {failed} failed")
    print(f"{'='*60}")



In [2]:

# Configuration (EDIT THESE)
start_path = "."              # Root directory to scan (. = current directory)
size_threshold_mb = 100       # File size threshold in MB
delete_originals = True       # Delete original files after compression?
dry_run = True                # Test run first? Set to False when ready

In [3]:
# Preview what will be compressed (DRY RUN)
print("🔍 PREVIEW MODE - See what will be compressed\n")
scan_and_compress(
    start_path=start_path,
    size_threshold_mb=size_threshold_mb,
    delete_originals=delete_originals,
    dry_run=True
)

🔍 PREVIEW MODE - See what will be compressed

🔍 Scanning . for files > 100MB...

✓ No files larger than 100MB found!


In [4]:
# Run the actual compression (after confirming dry run looks good)
# ONLY RUN THIS AFTER CHECKING CELL 4
dry_run = False  # Change from True to False
print("🗜️  RUNNING COMPRESSION\n")
scan_and_compress(
    start_path=start_path,
    size_threshold_mb=size_threshold_mb,
    delete_originals=delete_originals,
    dry_run=dry_run
)

🗜️  RUNNING COMPRESSION

🔍 Scanning . for files > 100MB...

✓ No files larger than 100MB found!
